# Quantra Engine - Phase 3: Historical Backtesting

This notebook demonstrates the historical engine capabilities of Quantra, evaluating both Markowitz and QAOA (Quantum) optimal portfolio allocations over out-of-sample data.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.data_loader import fetch_nifty50_prices, get_returns
from src.portfolio.markowitz import max_sharpe_portfolio
from src.quantum.benchmarker import QuantumClassicalBenchmark
from src.backtest.engine import BacktestEngine
from src.utils.visualizer import set_plot_style

set_plot_style()

## 1. Optimize Portfolios (In-Sample)
We use 2021-2022 to compute standard Markowitz and Quantum weights.

In [ ]:
tickers = [
    "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ICICIBANK.NS",
    "WIPRO.NS", "AXISBANK.NS", "KOTAKBANK.NS", "LT.NS", "SBIN.NS",
]

print("Fetching in-sample data...")
prices_is = fetch_nifty50_prices(tickers, "2021-01-01", "2022-01-01")
returns_is = get_returns(prices_is)
mu_is = returns_is.mean().values * 252
Sigma_is = returns_is.cov().values * 252

c_result = max_sharpe_portfolio(mu_is, Sigma_is, tickers)
c_weights = c_result["weights"]

benchmarker = QuantumClassicalBenchmark(returns_is, tickers)
q_result = benchmarker.run_quantum(n_assets_to_select=5, p_layers=1, max_iterations=20)
q_weights = q_result["weights"]

print("Classical Weights:", c_weights)
print("Quantum Weights:", q_weights)

## 2. Run Backtest (Out-of-Sample)
We test these fixed allocations through the volatile 2022-2024 period.

In [ ]:
engine_c = BacktestEngine(tickers, c_weights, "2022-01-01", "2024-01-01")
res_c = engine_c.run(rebalance_freq='monthly')

engine_q = BacktestEngine(tickers, q_weights, "2022-01-01", "2024-01-01")
res_q = engine_q.run(rebalance_freq='monthly')

print("Classical Out-of-Sample Sharpe:", res_c['metrics']['sharpe_ratio'])
print("Quantum Out-of-Sample Sharpe:", res_q['metrics']['sharpe_ratio'])

## 3. Equity Curve Visualization

In [ ]:
dates = list(res_c["equity_curve"].keys())
c_vals = list(res_c["equity_curve"].values())
q_vals = list(res_q["equity_curve"].values())

plt.figure(figsize=(10, 6))
plt.plot(pd.to_datetime(dates), c_vals, label='Classical Markowitz', color='cyan')
plt.plot(pd.to_datetime(dates), q_vals, label='QAOA Quantum Strategy', color='magenta')
plt.title('Out-of-Sample Equity Curves (2022 - 2024)')
plt.legend()
plt.show()